<a href="https://colab.research.google.com/github/minju0236/Hankyung-Bootcamp/blob/main/Day9_10_b_(260417)_%EC%83%81%ED%83%9C_%EA%B4%80%EB%A6%AC_Stale_%26_Mutation_%EC%8B%A4%EC%8A%B5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 상태 관리 계층 실습

In [ ]:
%%writefile scaling-state/node-server/server.js

const express = require('express');
const mysql = require('mysql2/promise');
const path = require('path');

const app = express();
const PORT = 3000;

app.use(express.json());
app.use(express.static(path.join(__dirname, 'static')));

const pool = mysql.createPool({
  host: 'localhost',
  user: 'testuser',
  password: '1234',
  database: 'testdb',
  dateStrings: true
});

// 거래 데이터 조회 (Server State)
app.get('/api/transactions', async (req, res) => {
  try {
    const [rows] = await pool.query(`
      SELECT id, name, amount
      FROM transactions
      ORDER BY id DESC
    `);

    await new Promise((resolve) => {
      setTimeout(resolve, 2000);
    });

    res.json(rows);
  } catch (e) {
    res.status(500).json({ message: '조회 실패' });
  }
});

// React Router 대응
app.get('/', (req, res) => {
  res.sendFile(path.join(__dirname, 'static', 'index.html'));
});

app.get('/*rest', (req, res) => {
  res.sendFile(path.join(__dirname, 'static', 'index.html'));
});

app.listen(PORT, () => {
  console.log('server running');
});

Writing scaling-state/node-server/server.js


In [ ]:
%%writefile scaling-state/react/src/main.jsx

import React from 'react';
import ReactDOM from 'react-dom/client';
import { QueryClient, QueryClientProvider } from '@tanstack/react-query';
import App from './App';

const queryClient = new QueryClient();

ReactDOM.createRoot(document.getElementById('root')).render(
  <QueryClientProvider client={queryClient}>
    <App />
  </QueryClientProvider>
);

Overwriting scaling-state/react/src/main.jsx


In [ ]:
%%writefile scaling-state/react/src/App.jsx

import Dashboard from './pages/Dashboard';

function App() {
  return <Dashboard />;
}

export default App;

Overwriting scaling-state/react/src/App.jsx


In [ ]:
%%writefile scaling-state/react/src/store.js

import { create } from 'zustand';

const useStore = create((set) => ({
  theme: 'light',
  toggleTheme: () =>
    set((state) => ({
      theme: state.theme === 'light' ? 'dark' : 'light'
    }))
}));

export default useStore;

Overwriting scaling-state/react/src/store.js


In [ ]:
%%writefile scaling-state/react/src/components/SearchBox.jsx

import { useState } from 'react';

function SearchBox() {
  const [keyword, setKeyword] = useState('');

  return (
    <input
      value={keyword}
      onChange={(e) => setKeyword(e.target.value)}
      placeholder="검색"
    />
  );
}

export default SearchBox;

Overwriting scaling-state/react/src/components/SearchBox.jsx


In [ ]:
%%writefile scaling-state/react/src/components/Timer.jsx

import { useEffect } from 'react';

function Timer() {
  useEffect(() => {
    const id = setInterval(() => {
      console.log('동작 중');
    }, 1000);

    return () => clearInterval(id);
  }, []);

  return <div>타이머 실행 중</div>;
}

export default Timer;

Overwriting scaling-state/react/src/components/Timer.jsx


In [ ]:
%%writefile scaling-state/react/src/components/ThemeButton.jsx

import useStore from '../store';

function ThemeButton() {
  const { theme, toggleTheme } = useStore();

  return (
    <button onClick={toggleTheme}>
      현재 테마: {theme}
    </button>
  );
}

export default ThemeButton;

Overwriting scaling-state/react/src/components/ThemeButton.jsx


In [ ]:
%%writefile scaling-state/react/src/components/TransactionList.jsx

import { useQuery } from '@tanstack/react-query';

async function fetchTransactions() {
  const res = await fetch('/api/transactions');

  if (!res.ok) {
    throw new Error('실패');
  }

  return res.json();
}

function TransactionList() {
  const { data, isLoading, isFetching, error } = useQuery({
    queryKey: ['transactions'],
    queryFn: fetchTransactions,
    staleTime: 5000
  });

  if (isLoading) return <div>로딩 중</div>;
  if (error) return <div>{error.message}</div>;

  return (
    <div>
      {isFetching && <div>갱신 중</div>}

      <ul>
        {data.map((item) => (
          <li key={item.id}>
            {item.name} - {item.amount}
          </li>
        ))}
      </ul>
    </div>
  );
}

export default TransactionList;

Overwriting scaling-state/react/src/components/TransactionList.jsx


In [ ]:
%%writefile scaling-state/react/src/pages/Dashboard.jsx

import SearchBox from '../components/SearchBox';
import Timer from '../components/Timer';
import ThemeButton from '../components/ThemeButton';
import TransactionList from '../components/TransactionList';

function Dashboard() {
  return (
    <div>
      <h1>상태 관리 계층 실습</h1>

      <SearchBox />
      <Timer />
      <ThemeButton />
      <TransactionList />
    </div>
  );
}

export default Dashboard;

Overwriting scaling-state/react/src/pages/Dashboard.jsx


# 상태 관리 아키텍처 실습

In [ ]:
%%writefile state-architecture/node-server/server.js

const express = require('express');
const mysql = require('mysql2/promise');
const path = require('path');

const app = express();
const PORT = 3000;

app.use(express.json());
app.use(express.static(path.join(__dirname, 'static')));

const pool = mysql.createPool({
  host: 'localhost',
  user: 'testuser',
  password: '1234',
  database: 'testdb',
  dateStrings: true
});

// 계좌 요약 조회
app.get('/api/account', async (req, res) => {
  try {
    const [[row]] = await pool.query(`
      SELECT id, owner_name, balance
      FROM accounts
      LIMIT 1
    `);

    res.json(row);
  } catch (e) {
    res.status(500).json({ message: '계좌 조회 실패' });
  }
});

// 거래 내역 조회
app.get('/api/transactions', async (req, res) => {
  try {
    const [rows] = await pool.query(`
      SELECT id, sender_name, amount, created_at
      FROM transactions
      ORDER BY id DESC
    `);

    await new Promise((resolve) => {
      setTimeout(resolve, 2000);
    });

    res.json(rows);
  } catch (e) {
    res.status(500).json({ message: '거래 내역 조회 실패' });
  }
});

// 거래 등록
app.post('/api/transactions', async (req, res) => {
  try {
    const { sender_name, amount } = req.body;

    if (!sender_name || !amount) {
      return res.status(400).json({ message: '입력값이 부족합니다.' });
    }

    await pool.query(
      `INSERT INTO transactions (sender_name, amount) VALUES (?, ?)`,
      [sender_name, Number(amount)]
    );

    res.json({ success: true });
  } catch (e) {
    res.status(500).json({ message: '거래 등록 실패' });
  }
});

// React Router 대응
app.get('/', (req, res) => {
  res.sendFile(path.join(__dirname, 'static', 'index.html'));
});

app.get('/*rest', (req, res) => {
  res.sendFile(path.join(__dirname, 'static', 'index.html'));
});

app.listen(PORT, () => {
  console.log('server running');
});

Overwriting state-architecture/node-server/server.js


In [ ]:
%%writefile state-architecture/react/src/main.jsx

import React from 'react';
import ReactDOM from 'react-dom/client';
import { QueryClient, QueryClientProvider } from '@tanstack/react-query';
import App from './App';

const queryClient = new QueryClient();

ReactDOM.createRoot(document.getElementById('root')).render(
  <QueryClientProvider client={queryClient}>
    <App />
  </QueryClientProvider>
);

Overwriting state-architecture/react/src/main.jsx


In [ ]:
%%writefile state-architecture/react/src/App.jsx

import BankingPage from './pages/BankingPage';

function App() {
  return <BankingPage />;
}

export default App;

Overwriting state-architecture/react/src/App.jsx


In [ ]:
%%writefile state-architecture/react/src/store.js

import { create } from 'zustand';

const useStore = create((set) => ({
  theme: 'light',
  form: {
    sender_name: '',
    amount: ''
  },

  toggleTheme: () =>
    set((state) => ({
      theme: state.theme === 'light' ? 'dark' : 'light'
    })),

  setForm: (key, value) =>
    set((state) => ({
      form: {
        ...state.form,
        [key]: value
      }
    }))
}));

export default useStore;

Writing state-architecture/react/src/store.js


In [ ]:
%%writefile state-architecture/react/src/api/bankingApi.js

export async function fetchAccount() {
  const res = await fetch('/api/account');

  if (!res.ok) {
    throw new Error('계좌 조회 실패');
  }

  return res.json();
}

export async function fetchTransactions() {
  const res = await fetch('/api/transactions');

  if (!res.ok) {
    throw new Error('거래 내역 조회 실패');
  }

  return res.json();
}

export async function createTransaction(data) {
  const res = await fetch('/api/transactions', {
    method: 'POST',
    headers: {
      'Content-Type': 'application/json'
    },
    body: JSON.stringify(data)
  });

  if (!res.ok) {
    const errorData = await res.json();
    throw new Error(errorData.message || '거래 등록 실패');
  }

  return res.json();
}

Writing state-architecture/react/src/api/bankingApi.js


In [ ]:
%%writefile state-architecture/react/src/hooks/useBankingState.js

import useStore from '../store';
import { useMutation, useQuery, useQueryClient } from '@tanstack/react-query';
import {
  createTransaction,
  fetchAccount,
  fetchTransactions
} from '../api/bankingApi';

export function useBankingState() {
  const queryClient = useQueryClient();
  const { theme, toggleTheme, form, setForm } = useStore();

  const accountQuery = useQuery({
    queryKey: ['account'],
    queryFn: fetchAccount,
    staleTime: 0
  });

  const transactionsQuery = useQuery({
    queryKey: ['transactions'],
    queryFn: fetchTransactions,
    staleTime: 0
  });

  const createMutation = useMutation({
    mutationFn: createTransaction,
    onSuccess: () => {
      queryClient.invalidateQueries({ queryKey: ['transactions'] });
    }
  });

  function handleChange(event) {
    setForm(event.target.name, event.target.value);
  }

  function handleSubmit(event) {
    event.preventDefault();

    createMutation.mutate({
      sender_name: form.sender_name,
      amount: Number(form.amount)
    });
  }

  return {
    theme,
    toggleTheme,
    form,
    handleChange,
    handleSubmit,
    accountQuery,
    transactionsQuery,
    createMutation
  };
}

Overwriting state-architecture/react/src/hooks/useBankingState.js


In [ ]:
%%writefile state-architecture/react/src/components/ThemeToggle.jsx

function ThemeToggle({ theme, toggleTheme }) {
  return (
    <button onClick={toggleTheme}>
      현재 테마: {theme}
    </button>
  );
}

export default ThemeToggle;

Writing state-architecture/react/src/components/ThemeToggle.jsx


In [ ]:
%%writefile state-architecture/react/src/components/BankingForm.jsx

function BankingForm({ form, handleChange, handleSubmit, isPending, error }) {
  return (
    <form onSubmit={handleSubmit}>
      <input
        name="sender_name"
        value={form.sender_name}
        onChange={handleChange}
        placeholder="이름"
      />

      <input
        name="amount"
        value={form.amount}
        onChange={handleChange}
        placeholder="금액"
      />

      <button type="submit">거래 등록</button>

      {isPending && <div>등록 중</div>}
      {error && <div>{error.message}</div>}
    </form>
  );
}

export default BankingForm;

Writing state-architecture/react/src/components/BankingForm.jsx


In [ ]:
%%writefile state-architecture/react/src/components/AccountSummary.jsx

function AccountSummary({ account, isLoading, error }) {
  if (isLoading) return <div>계좌 로딩 중</div>;
  if (error) return <div>{error.message}</div>;

  return (
    <div>
      <h3>계좌 요약</h3>
      <p>예금주: {account.owner_name}</p>
      <p>잔액: {account.balance}</p>
    </div>
  );
}

export default AccountSummary;

Writing state-architecture/react/src/components/AccountSummary.jsx


In [ ]:
%%writefile state-architecture/react/src/components/TransactionList.jsx

function TransactionList({ transactions, isLoading, isFetching, error }) {
  if (isLoading) return <div>거래 목록 로딩 중</div>;
  if (error) return <div>{error.message}</div>;

  return (
    <div>
      {isFetching && <div>최신 데이터 갱신 중</div>}

      <ul>
        {transactions.map((item) => (
          <li key={item.id}>
            {item.sender_name} - {item.amount} - {item.created_at}
          </li>
        ))}
      </ul>
    </div>
  );
}

export default TransactionList;

Writing state-architecture/react/src/components/TransactionList.jsx


In [ ]:
%%writefile state-architecture/react/src/pages/BankingPage.jsx

import ThemeToggle from '../components/ThemeToggle';
import BankingForm from '../components/BankingForm';
import AccountSummary from '../components/AccountSummary';
import TransactionList from '../components/TransactionList';
import { useBankingState } from '../hooks/useBankingState';

function BankingPage() {
  const {
    theme,
    toggleTheme,
    form,
    handleChange,
    handleSubmit,
    accountQuery,
    transactionsQuery,
    createMutation
  } = useBankingState();

  return (
    <div>
      <h1>상태 관리 아키텍처 실습</h1>

      <ThemeToggle theme={theme} toggleTheme={toggleTheme} />

      <BankingForm
        form={form}
        handleChange={handleChange}
        handleSubmit={handleSubmit}
        isPending={createMutation.isPending}
        error={createMutation.error}
      />

      <AccountSummary
        account={accountQuery.data}
        isLoading={accountQuery.isLoading}
        error={accountQuery.error}
      />

      <TransactionList
        transactions={transactionsQuery.data || []}
        isLoading={transactionsQuery.isLoading}
        isFetching={transactionsQuery.isFetching}
        error={transactionsQuery.error}
      />
    </div>
  );
}

export default BankingPage;

Writing state-architecture/react/src/pages/BankingPage.jsx
